<a href="https://colab.research.google.com/github/bautil00/Hackathon-Project-/blob/main/KPI_Performance_Indicator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# KPI COMPARISON DASHBOARD - GOOGLE COLAB VERSION
# ============================================================

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import json
from datetime import datetime

# Define KPI benchmarks
kpi_benchmarks = {
    'Business Goals Met': {'target': 80, 'unit': '%', 'direction': 'up'},
    'On-Schedule Delivery': {'target': 60, 'unit': '%', 'direction': 'up'},
    'On-Budget Delivery': {'target': 70, 'unit': '%', 'direction': 'up'},
    'Failure Rate': {'target': 10, 'unit': '%', 'direction': 'down'},
    'Benefits Realized': {'target': 70, 'unit': '%', 'direction': 'up'},
    'Scope Changes (Post-Freeze)': {'target': 15, 'unit': '%', 'direction': 'down'},
    'Cycle Time Trend': {'target': 75, 'unit': ' score', 'direction': 'up'},
    'Risk Burn-Down': {'target': 85, 'unit': '%', 'direction': 'up'},
    'Stakeholder Satisfaction': {'target': 80, 'unit': '%', 'direction': 'up'},
    'Success Factors Tracked': {'target': 9, 'unit': ' count', 'direction': 'up'}
}

# Create input widgets
print("=" * 70)
print("📊 KPI COMPARISON DASHBOARD - Google Colab")
print("=" * 70)
print("\nEnter your current metrics in the boxes below:\n")

# Create input fields
inputs = {}
for kpi, info in kpi_benchmarks.items():
    inputs[kpi] = widgets.FloatText(
        value=0.0,
        description=f"{kpi[:25]}:",
        description_width='initial',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px')
    )
    display(inputs[kpi])

# Submit button
submit_button = widgets.Button(
    description='📊 Generate KPI Dashboard',
    button_style='success',
    layout=widgets.Layout(width='300px', margin='20px 0px')
)

output_area = widgets.Output()

def generate_dashboard(b):
    with output_area:
        clear_output(wait=True)

        # Collect user inputs
        user_inputs = {}
        for kpi, widget in inputs.items():
            user_inputs[kpi] = widget.value

        # Calculate results
        results = []
        for kpi, info in kpi_benchmarks.items():
            current = user_inputs[kpi]
            target = info['target']
            direction = info['direction']

            # Calculate percentage of target
            if direction == 'up':
                pct_of_target = (current / target) * 100 if target > 0 else 0
                gap_pct = pct_of_target - 100
            else:
                if current > 0:
                    pct_of_target = (target / current) * 100
                else:
                    pct_of_target = 0
                gap_pct = pct_of_target - 100

            # Determine rating
            if gap_pct >= 20:
                rating = "EXCELLENT"
            elif gap_pct >= 5:
                rating = "GOOD"
            elif gap_pct >= -5:
                rating = "NEUTRAL"
            elif gap_pct >= -20:
                rating = "NEEDS IMPROVEMENT"
            else:
                rating = "BAD"

            # Format comparison text
            if direction == 'up':
                if current > target:
                    comparison = f"{current}{info['unit']} > {target}{info['unit']} (+{current-target:.1f})"
                elif current < target:
                    comparison = f"{current}{info['unit']} < {target}{info['unit']} ({current-target:+.1f})"
                else:
                    comparison = f"{current}{info['unit']} = {target}{info['unit']}"
            else:
                if current <= target:
                    comparison = f"{current}{info['unit']} ≤ {target}{info['unit']} (Met target)"
                else:
                    comparison = f"{current}{info['unit']} > {target}{info['unit']} (+{current-target:.1f})"

            results.append({
                'KPI': kpi,
                'User Filled': f"{current}{info['unit']}",
                'Comparative Result': f"{rating} | {comparison}"
            })

        # Display results in table format
        print("\n" + "=" * 70)
        print("📋 KPI COMPARISON RESULTS")
        print("=" * 70)
        print(f"\n{'KPI':<35} {'User Filled':<15} {'Comparative Result':<50}")
        print("-" * 100)

        for row in results:
            print(f"{row['KPI']:<35} {row['User Filled']:<15} {row['Comparative Result']:<50}")

        # Summary statistics
        print("\n" + "=" * 70)
        print("📊 SUMMARY STATISTICS")
        print("=" * 70)

        rating_counts = {'EXCELLENT': 0, 'GOOD': 0, 'NEUTRAL': 0, 'NEEDS IMPROVEMENT': 0, 'BAD': 0}
        for row in results:
            for rating in rating_counts:
                if rating in row['Comparative Result']:
                    rating_counts[rating] += 1
                    break

        print(f"\n  🏆 EXCELLENT: {rating_counts['EXCELLENT']} KPI(s)")
        print(f"  ✅ GOOD: {rating_counts['GOOD']} KPI(s)")
        print(f"  ⚖️ NEUTRAL: {rating_counts['NEUTRAL']} KPI(s)")
        print(f"  ⚠️ NEEDS IMPROVEMENT: {rating_counts['NEEDS IMPROVEMENT']} KPI(s)")
        print(f"  ❌ BAD: {rating_counts['BAD']} KPI(s)")

        # Overall assessment
        if rating_counts['BAD'] >= 3:
            overall = "🔴 CRITICAL - Immediate action required"
        elif rating_counts['NEEDS IMPROVEMENT'] >= 4:
            overall = "🟡 WARNING - Significant improvement needed"
        elif rating_counts['EXCELLENT'] >= 5:
            overall = "🟢 STRONG - Excellent performance across most KPIs"
        elif rating_counts['GOOD'] >= 6:
            overall = "🔵 GOOD - Solid performance with minor gaps"
        else:
            overall = "⚪ MIXED - Some strengths, some weaknesses"

        print(f"\n  Overall Assessment: {overall}")

        # Priority KPIs
        priority_kpis = [row for row in results if 'BAD' in row['Comparative Result'] or 'NEEDS IMPROVEMENT' in row['Comparative Result']]
        if priority_kpis:
            print(f"\n  🎯 Top Priority KPIs to Address:")
            for kpi in priority_kpis[:3]:
                print(f"     • {kpi['KPI']}")

        # Generate HTML for Figma/PDF
        generate_html_for_figma(results)

submit_button.on_click(generate_dashboard)
display(submit_button, output_area)

def generate_html_for_figma(results):
    """Generates HTML dashboard for Figma/PDF export"""

    html_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>KPI Performance Dashboard</title>
    <style>
        * {{ margin: 0; padding: 0; box-sizing: border-box; }}
        body {{
            font-family: 'Inter', -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
            background: #F0F2F5;
            padding: 40px 20px;
        }}
        .dashboard {{
            max-width: 1200px;
            margin: 0 auto;
            background: white;
            border-radius: 16px;
            box-shadow: 0 4px 12px rgba(0,0,0,0.1);
            overflow: hidden;
        }}
        .header {{
            background: #2C3E50;
            color: white;
            padding: 30px 40px;
        }}
        .header h1 {{ font-size: 28px; margin-bottom: 8px; }}
        .table-container {{ padding: 20px 40px; overflow-x: auto; }}
        table {{ width: 100%; border-collapse: collapse; }}
        th {{
            background: #F8F9FA;
            padding: 15px;
            text-align: left;
            border-bottom: 2px solid #E0E0E0;
        }}
        td {{ padding: 15px; border-bottom: 1px solid #F0F0F0; }}
        .status-badge {{
            display: inline-block;
            padding: 5px 12px;
            border-radius: 20px;
            font-size: 12px;
            font-weight: 600;
            color: white;
        }}
        .status-EXCELLENT {{ background: #2ECC71; }}
        .status-GOOD {{ background: #58D68D; }}
        .status-NEUTRAL {{ background: #F39C12; }}
        .status-NEEDS-IMPROVEMENT {{ background: #E67E22; }}
        .status-BAD {{ background: #E74C3C; }}
        .stats {{
            display: grid;
            grid-template-columns: repeat(5, 1fr);
            gap: 20px;
            padding: 30px 40px;
            background: #F8F9FA;
        }}
        .stat-card {{
            background: white;
            padding: 20px;
            border-radius: 12px;
            text-align: center;
        }}
        .stat-value {{ font-size: 32px; font-weight: bold; margin: 10px 0; }}
        .assessment {{
            background: #667eea;
            color: white;
            padding: 30px 40px;
            text-align: center;
        }}
        @media print {{
            body {{ background: white; padding: 0; }}
            .dashboard {{ box-shadow: none; }}
            .status-badge {{ print-color-adjust: exact; -webkit-print-color-adjust: exact; }}
        }}
    </style>
</head>
<body>
    <div class="dashboard">
        <div class="header">
            <h1>📊 KPI Performance Dashboard</h1>
            <p>Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}</p>
        </div>
        <div class="table-container">
            <table>
                <thead>
                    <tr>
                        <th>KPI</th>
                        <th>Your Value</th>
                        <th>Target</th>
                        <th>Status</th>
                        <th>Comparison</th>
                    </tr>
                </thead>
                <tbody>
"""

    for row in results:
        comp_parts = row['Comparative Result'].split(' | ')
        rating = comp_parts[0].strip()
        comparison = comp_parts[1] if len(comp_parts) > 1 else ''

        # Extract target
        if '>' in comparison or '<' in comparison or '≤' in comparison:
            target = comparison.split()[-1] if comparison.split() else 'N/A'
        else:
            target = 'N/A'

        html_content += f"""
                    <tr>
                        <td><strong>{row['KPI']}</strong></td>
                        <td>{row['User Filled']}</td>
                        <td>{target}</td>
                        <td><span class="status-badge status-{rating.replace(' ', '-')}">{rating}</span></td>
                        <td style="color: #666;">{comparison}</td>
                    </tr>
"""

    # Calculate stats
    rating_counts = {'EXCELLENT': 0, 'GOOD': 0, 'NEUTRAL': 0, 'NEEDS IMPROVEMENT': 0, 'BAD': 0}
    for row in results:
        for rating in rating_counts:
            if rating in row['Comparative Result']:
                rating_counts[rating] += 1
                break

    total_kpis = len(results)
    success_rate = ((rating_counts['EXCELLENT'] + rating_counts['GOOD']) / total_kpis) * 100

    # Overall assessment
    if rating_counts['BAD'] >= 3:
        overall_icon = "🔴"
        overall_text = "CRITICAL - Immediate action required"
    elif rating_counts['NEEDS IMPROVEMENT'] >= 4:
        overall_icon = "🟡"
        overall_text = "WARNING - Significant improvement needed"
    elif rating_counts['EXCELLENT'] >= 5:
        overall_icon = "🟢"
        overall_text = "STRONG - Excellent performance"
    elif rating_counts['GOOD'] >= 6:
        overall_icon = "🔵"
        overall_text = "GOOD - Solid performance"
    else:
        overall_icon = "⚪"
        overall_text = "MIXED - Some strengths, some weaknesses"

    html_content += f"""
                </tbody>
            </table>
        </div>
        <div class="stats">
            <div class="stat-card"><div class="stat-label">🏆 EXCELLENT</div><div class="stat-value" style="color:#2ECC71">{rating_counts['EXCELLENT']}</div></div>
            <div class="stat-card"><div class="stat-label">✅ GOOD</div><div class="stat-value" style="color:#58D68D">{rating_counts['GOOD']}</div></div>
            <div class="stat-card"><div class="stat-label">⚖️ NEUTRAL</div><div class="stat-value" style="color:#F39C12">{rating_counts['NEUTRAL']}</div></div>
            <div class="stat-card"><div class="stat-label">⚠️ NEEDS IMPROVEMENT</div><div class="stat-value" style="color:#E67E22">{rating_counts['NEEDS IMPROVEMENT']}</div></div>
            <div class="stat-card"><div class="stat-label">❌ BAD</div><div class="stat-value" style="color:#E74C3C">{rating_counts['BAD']}</div></div>
        </div>
        <div class="assessment">
            <h3>{overall_icon} {overall_text}</h3>
            <p>Success Rate: {success_rate:.0f}% of KPIs meeting or exceeding targets</p>
        </div>
    </div>
</body>
</html>
"""

    # Save HTML file in Colab
    with open('/content/kpi_dashboard_figma.html', 'w', encoding='utf-8') as f:
        f.write(html_content)

    print("\n" + "=" * 70)
    print("✅ HTML DASHBOARD GENERATED FOR FIGMA!")
    print("=" * 70)
    print("\n📁 File saved in Colab: kpi_dashboard_figma.html")

    # Provide download link
    from google.colab import files
    files.download('/content/kpi_dashboard_figma.html')

    print("\n📌 TO CREATE PDF FOR FIGMA:")
    print("   1. Download the HTML file (auto-downloading now)")
    print("   2. Open in Chrome browser")
    print("   3. Press Ctrl+P (Cmd+P on Mac)")
    print("   4. Save as PDF → Import into Figma")

📊 KPI COMPARISON DASHBOARD - Google Colab

Enter your current metrics in the boxes below:



FloatText(value=0.0, description='Business Goals Met:', layout=Layout(width='400px'), style=DescriptionStyle(d…

FloatText(value=0.0, description='On-Schedule Delivery:', layout=Layout(width='400px'), style=DescriptionStyle…

FloatText(value=0.0, description='On-Budget Delivery:', layout=Layout(width='400px'), style=DescriptionStyle(d…

FloatText(value=0.0, description='Failure Rate:', layout=Layout(width='400px'), style=DescriptionStyle(descrip…

...

FloatText(value=0.0, description='Risk Burn-Down:', layout=Layout(width='400px'), style=DescriptionStyle(descr…

FloatText(value=0.0, description='Stakeholder Satisfaction:', layout=Layout(width='400px'), style=DescriptionS…

FloatText(value=0.0, description='Success Factors Tracked:', layout=Layout(width='400px'), style=DescriptionSt…

Button(button_style='success', description='📊 Generate KPI Dashboard', layout=Layout(margin='20px 0px', width=…

Output()